## TOOLS


Models can call tools to handle some tasks, these tools can be functions

In [4]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain.tools import tool

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

response = model.invoke("What is the weather in London?")
response

AIMessage(content="<think>\nOkay, the user is asking for the current weather in London. Let me check how to get that information.\n\nFirst, I remember that I can't access real-time data, so I can't provide the current weather directly. I should inform the user about that limitation.\n\nI need to let them know they can check a weather website or app for the latest updates. Maybe suggest some reliable sources like the UK Met Office or Weather.com.\n\nAlso, maybe mention typical weather patterns in London as a general idea, like being temperate with frequent rain and mild temperatures. But emphasize that it's not real-time.\n\nMake sure the response is helpful and guides them to where they can find accurate information. Keep it friendly and clear.\n</think>\n\nI currently don’t have access to real-time weather data, but you can check the latest weather in London using a trusted weather service like the **UK Met Office** ([www.metoffice.gov.uk](https://www.metoffice.gov.uk)), **Weather.com

In [5]:
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # Placeholder implementation - replace with actual weather API call
    return f"The weather in {location} is currently sunny."

model_with_tools = model.bind_tools([get_weather])

In [6]:
response = model_with_tools.invoke("What is the weather in London?")

for tool_call in response.tool_calls :
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

Tool: get_weather
Args: {'location': 'London'}


## Tools execution loop

In [7]:
messages = [{"role": "user", "content": "What is the weather in London?"},] # list of messages to be sent to the model, each message is a dictionary with "role" and "content" keys
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message) # used to append an object to the end of a list

# step 2: execute tools and collect results
for tool_call in ai_message.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

#step 3 : send tool results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in London is currently sunny. A perfect day to enjoy outdoor activities! 🌞
